# Three evaluation protocols, and the leaks that defeat them

Hold-out, K-fold, and iterated K-fold — with the four ways a test score gets quietly contaminated.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 5 — Fundamentals of Machine Learning](../../../course-web-slides/ch05/index.html) &nbsp;·&nbsp; **Section:** 03 — Evaluating machine learning models

---

## A common-sense baseline first

In [ ]:
import numpy as np
from keras.datasets import mnist

(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

print("random guessing:      ", f"{1/10:.3f}")
print("most common class:    ", f"{np.bincount(y).max()/len(y):.3f}")

# A genuinely trivial model, as a floor to beat.
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=200, n_jobs=-1).fit(x[:10000], y[:10000])
print("logistic regression:  ", f"{lr.score(xt, yt):.3f}")

**If a deep model cannot beat that third number, something is wrong** — and finding out now costs a minute rather than a week.

## Hold-out validation

In [ ]:
num_validation_samples = 10000
np.random.seed(0)
idx = np.random.permutation(len(x))
x_sh, y_sh = x[idx], y[idx]

validation_x, training_x = x_sh[:num_validation_samples], x_sh[num_validation_samples:]
validation_y, training_y = y_sh[:num_validation_samples], y_sh[num_validation_samples:]

print(f"train {len(training_x)}   validation {len(validation_x)}   test {len(xt)}")

> **Note** — Simplest, and it needs enough data that the validation split is statistically meaningful. With a few hundred samples it is not, which is why chapter 4's housing example reached for K-fold.

## K-fold, and why the spread matters more than the mean

In [ ]:
import keras
from keras import layers

def small_model():
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

k, n = 5, 5000                    # a deliberately small subset
xs, ys = x[:n], y[:n]
fold = n // k
scores = []
for i in range(k):
    vx, vy = xs[i*fold:(i+1)*fold], ys[i*fold:(i+1)*fold]
    tx = np.concatenate([xs[:i*fold], xs[(i+1)*fold:]])
    ty = np.concatenate([ys[:i*fold], ys[(i+1)*fold:]])
    m = small_model()
    m.fit(tx, ty, epochs=8, batch_size=64, verbose=0)
    scores.append(m.evaluate(vx, vy, verbose=0)[1])

print("folds:", [f"{s:.4f}" for s in scores])
print(f"mean {np.mean(scores):.4f}   std {np.std(scores):.4f}   "
      f"spread {max(scores)-min(scores):.4f}")

Report the spread alongside the mean. **A one-point improvement inside a two-point spread is not an improvement** — it is a different fold.

## Leak 1: preprocessing fitted before the split

In [ ]:
from sklearn.preprocessing import StandardScaler

# WRONG: statistics computed over everything, including validation.
bad = StandardScaler().fit(np.vstack([training_x, validation_x]))

# RIGHT: statistics from training data only.
good = StandardScaler().fit(training_x)

print("mean of feature 400, all data:  ", f"{bad.mean_[400]:.6f}")
print("mean of feature 400, train only:", f"{good.mean_[400]:.6f}")
print("difference:", f"{abs(bad.mean_[400]-good.mean_[400]):.6f}")

The difference is tiny here and the principle is not. Anything **fitted** on data — a scaler, a vocabulary, a PCA basis, `TextVectorization.adapt()` — must see only the training split.

## Leak 2: duplicates across the split

In [ ]:
# Simulate a dataset where some samples appear twice.
dup = np.vstack([x[:2000], x[:500]])
dup_y = np.concatenate([y[:2000], y[:500]])

perm = np.random.permutation(len(dup))
dup, dup_y = dup[perm], dup_y[perm]

train_part, val_part = dup[:2000], dup[2000:]
# Are any validation samples byte-identical to a training sample?
shared = sum(any(np.array_equal(v, t) for t in train_part) for v in val_part[:50])
print(f"{shared} of the first 50 validation samples also appear in training")

> ⚠️ **Redundancy is the leak nobody looks for.** A scraped dataset with duplicate records will put the same sample on both sides of the split, and your validation score becomes partly a training score.

## Leak 3: temporal data split at random

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(400)
series = np.cumsum(np.random.default_rng(1).normal(size=400)) + 20

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
r = np.random.default_rng(0).permutation(400)
a1.scatter(t[r[:320]], series[r[:320]], s=8, label="train")
a1.scatter(t[r[320:]], series[r[320:]], s=8, label="validation")
a1.set_title("WRONG — random split on a timeseries"); a1.legend()

a2.plot(t[:320], series[:320], lw=1.4, label="train")
a2.plot(t[320:], series[320:], lw=1.4, label="validation")
a2.set_title("RIGHT — validation is posterior"); a2.legend()
plt.tight_layout(); plt.show()

In the left panel the model interpolates between points it has already seen on both sides. **You would be predicting the past from the future**, and the score would be excellent and worthless.

## Leak 4: the test set used more than once

The subtlest one, and it has no code. Every time you look at the test score and change something, a little information moves from the test set into your model — through you.

Chapter 18 gives this a name in the context of automated tuning (**validation-set overfitting**), but it applies just as much to a human iterating by hand. ==A test set is a one-shot instrument.==

---

## What to take away

- Compute a common-sense baseline before believing any score.
- Report the **spread** across folds, not only the mean.
- Anything fitted on data must be fitted on the training split only.
- Duplicates, temporal splits, and repeated test-set use are the three leaks that produce excellent, worthless numbers.